
El departamento de ventas ha decidido poner una pantalla con un kpi que representa el estado del departamento. Para ello, uno de los empleados a vibecodeado un pequeño codigo en python que se conectará, leeraá la tabla workspace.gold.private_sales, hará una query sumando todas las ventas de lo que llevamos de año y lo publicará en una pantalla. Ya tienen el código listo y van a probarlo con las credenciales del jefe del departamento.

In [0]:
%sql
create schema workspace.gold

In [0]:
%sql
create table if not exists workspace.gold.private_sales (
    date string,
    sales decimal(8,2),
    store string
);

INSERT INTO workspace.gold.private_sales (date, sales, store) VALUES
-- 2025 Records
('2025-01-05', 1250.50, 'Store_North'),
('2025-01-12', 840.20, 'Store_South'),
('2025-01-18', 2100.00, 'Store_East'),
('2025-01-25', 450.75, 'Store_West'),
('2025-02-02', 1780.30, 'Store_North'),
('2025-02-14', 3200.00, 'Store_East'),
('2025-02-20', 910.15, 'Store_South'),
('2025-03-01', 1340.60, 'Store_West'),
('2025-03-15', 2450.80, 'Store_North'),
('2025-03-28', 1120.00, 'Store_East'),
('2025-04-04', 670.45, 'Store_South'),
('2025-04-12', 1890.90, 'Store_West'),
('2025-05-01', 2990.00, 'Store_North'),
('2025-05-18', 810.25, 'Store_East'),
('2025-06-05', 1430.50, 'Store_South'),
('2025-06-22', 2150.10, 'Store_West'),
('2025-07-04', 3500.00, 'Store_North'),
('2025-07-19', 980.75, 'Store_East'),
('2025-08-11', 1620.40, 'Store_South'),
('2025-08-29', 2040.80, 'Store_West'),
('2025-09-15', 1150.00, 'Store_North'),
('2025-10-02', 2780.60, 'Store_East'),
('2025-10-20', 890.30, 'Store_South'),
('2025-11-14', 3100.25, 'Store_West'),
('2025-12-24', 4250.90, 'Store_North'),

-- 2026 Records
('2026-01-03', 1310.40, 'Store_South'),
('2026-01-15', 2250.80, 'Store_East'),
('2026-01-28', 760.15, 'Store_West'),
('2026-02-05', 1950.00, 'Store_North'),
('2026-02-14', 3400.50, 'Store_East'),
('2026-02-22', 1050.20, 'Store_South'),
('2026-03-08', 1580.90, 'Store_West'),
('2026-03-19', 2600.30, 'Store_North'),
('2026-03-31', 920.60, 'Store_East'),
('2026-04-10', 1740.10, 'Store_South'),
('2026-04-25', 2180.75, 'Store_West'),
('2026-05-02', 3050.00, 'Store_North'),
('2026-05-16', 880.45, 'Store_East'),
('2026-05-30', 1490.80, 'Store_South'),
('2026-06-12', 2310.25, 'Store_West'),
('2026-06-28', 3650.00, 'Store_North'),
('2026-07-09', 1020.50, 'Store_East'),
('2026-07-21', 1830.15, 'Store_South'),
('2026-08-04', 2120.90, 'Store_West'),
('2026-08-15', 1280.00, 'Store_North'),
('2026-08-30', 2890.70, 'Store_East'),
('2026-09-10', 940.35, 'Store_South'),
('2026-09-24', 3250.60, 'Store_West'),
('2026-10-05', 1670.00, 'Store_North'),
('2026-10-18', 2410.85, 'Store_East');

Ahora que tenemos el sample, vamos a empezar a gestionar las malas prácticas:
1) Usar credenciales de personas en procesos, aunque sean de personas clave, es una mala práctica. La gente se va y las cosas se rompen. Usaremos un service principal.
2) Cuando tenemos que asignar permisos, debemos seguir el principio del least privilege. 

In [0]:
%sql
-- Para minimizar los accesos del service principal, vamos a crear una vista

create view  workspace.gold.sales_kpi as (
    select sum(sales) as kpi from workspace.gold.private_sales where date_trunc('yyyy', date) = date_trunc('yyyy', current_date())
)

In [0]:
%sql
select * from workspace.gold.sales_kpi

In [0]:
%sql
/* Ahora le daremos acceso al service principal (en realidad deberiamos usar roles, pero para simplificar usaremos el service principal)*/

GRANT USE CATALOG ON CATALOG workspace TO `39f6e1b0-c39c-4658-845a-5c10cc01b5b1`;
GRANT USE SCHEMA ON SCHEMA workspace.gold TO `39f6e1b0-c39c-4658-845a-5c10cc01b5b1`;
GRANT SELECT ON VIEW workspace.gold.sales_kpi TO `39f6e1b0-c39c-4658-845a-5c10cc01b5b1`;


Now we have a service principal that can reach our view,